# Bhoonidhi SDK — query

Search the portal and manage saved queries from Python. Every method
matches a `bhd query` command.

Runs live. `create`, `refresh`, and `download` reach the portal and need
a login; `list`, `show`, `fork`, `rename`, `rm` work on the local query
store. Enter your real credentials when prompted.

## 1. Log in

In [1]:
import getpass
from bhoonidhi_downloader.sdk import BhoonidhiClient, BhoonidhiError

client = BhoonidhiClient()
client.login(input("Bhoonidhi username: "), getpass.getpass("Bhoonidhi password: "))
print("authenticated:", client.is_authenticated)

authenticated: True


## 2. Create a query

`client.query.create(...)` matches `bhd query create`. Bounding box (minx, maxx, miny, maxy), date range, satellite/sensor. Returns the saved query, or None if nothing matched.

In [2]:
from datetime import datetime

query = client.query.create(
    91.77, 92.0, 25.496, 25.695,
    datetime(2025, 12, 1), datetime(2025, 12, 30),
    satellite="Sentinel-2A", sensor="MSI",
)
if query is None:
    print("no scenes matched")
else:
    print("slug:  ", query.slug)
    print("name:  ", query.name)
    print("scenes:", len(query.scenes))

Output()

slug:   silent-vale
name:   Sentinel-2A MSI scenes, Dec 2025
scenes: 3


## 3. List saved queries

`client.query.list()` matches `bhd query list`.

In [3]:
for q in client.query.list()[:10]:
    print(f"{q.slug:16s} {len(q.scenes):>4} scenes  {q.satellite}")

amber-isle         28 scenes  ResourceSat-2A
azure-grove         3 scenes  Sentinel-2A
bold-ridge          8 scenes  ResourceSat-2A
brisk-falcon      500 scenes  EOS-04
brisk-hollow       99 scenes  JPSS1
brisk-thicket     398 scenes  ResourceSat-2A
calm-cliff          4 scenes  CartoSat-2S
deep-cape           3 scenes  Sentinel-2A
deep-harbor       829 scenes  CartoSat-2S
gentle-harbor     500 scenes  EOS-04


## 4. Show one query's scenes

`client.query.show(slug)` matches `bhd query show`. Returns the query; each scene is a dict of portal fields.

In [4]:
slug = query.slug if query else client.query.list()[0].slug
q = client.query.show(slug)
print("showing:", q.slug, "|", len(q.scenes), "scenes")
for s in q.scenes[:5]:
    print(" ", s.get("ID"), "|", s.get("DOP"))

showing: silent-vale | 3 scenes
  SEN2A_MSI_zzz_18DEC2025_133_T46RDP_ESA_STUBBAOJD_20251218T065911 | 18-Dec-2025
  SEN2A_MSI_zzz_18DEC2025_133_T46RDP_ESA_STUBTAOJD_20251218T052031 | 18-Dec-2025
  SEN2A_MSI_zzz_18DEC2025_133_T46RCP_ESA_STUBBAOJD_20251218T065911 | 18-Dec-2025


## 5. Rename

`client.query.rename(...)` matches `bhd query rename`.

In [5]:
renamed = client.query.rename(slug, description="Edited from the SDK notebook")
print("name:", renamed.name)
print("desc:", renamed.description)

name: Sentinel-2A MSI scenes, Dec 2025
desc: Edited from the SDK notebook


## 6. Fork

`client.query.fork(slug)` matches `bhd query fork` — clones scenes under a new slug, no re-query.

In [6]:
fork = client.query.fork(slug, name="Notebook fork")
print("forked ->", fork.slug, "|", len(fork.scenes), "scenes")

forked -> amber-forest | 3 scenes


## 7. Refresh

`client.query.refresh(slug)` matches `bhd query refresh`. Returns `(query, added_count)`; `added_count` is None if already up to date.

In [7]:
refreshed, added = client.query.refresh(slug)
print("added:", added, "| total now:", len(refreshed.scenes))

Output()

added: 47 | total now: 50


## 8. Download (dry run first)

There is no `--dry-run` flag on the SDK method — instead preview with the download preview helper, then call `download` for real.

In [8]:
from pathlib import Path
from bhoonidhi_downloader.core.download import build_preview
from bhoonidhi_downloader.core.query.command import resolve_scene_selection

scenes = resolve_scene_selection(client.query.show(slug).scenes, None)
previews = build_preview(scenes, Path("/tmp/bhd_downloads"))
from collections import Counter
print(Counter(p.status for p in previews))

Counter({'would_download': 46, 'already_here': 4})


## 9. Download for real

`client.query.download(slug, out, on_progress=...)` matches `bhd query download`. Priced/on-order scenes are skipped. Uncomment to run — it fetches real files.

In [10]:
def show_progress(scene_id, done, total):
    pct = f"{done/total*100:4.0f}%" if total else "  ? "
    print(f"{pct}  {scene_id}", end="\r")

# select is a list of scene indices (1-based) and/or scene IDs:
#   select=[1, 2, 3]        -> first three scenes
#   select=["RAW12JUL..."]  -> a specific scene by ID
# omit select to download the whole query.
outcomes = client.query.download(slug, "/tmp/bhd_downloads", on_progress=show_progress, select=[1,32,3,4])
from collections import Counter
print(Counter(o.status for o in outcomes))

Counter({'already_downloaded': 3, 'downloaded': 1})AOJD_20260517T080805


## 10. Clean up

`client.query.rm(slug)` matches `bhd query rm`.

In [11]:
client.query.rm(fork.slug)
print("removed fork:", fork.slug)

removed fork: amber-forest
